# Single-TSFM Fine-Grained Error Analysis (English Visualization Edition)

This notebook provides:
1. robust outlier-guarded analysis data,
2. fully English chart titles/labels,
3. sorted bucket intervals for all bucket-based plots,
4. detailed per-figure interpretation guides,
5. a transparent workflow to search for a bucket-based classification scheme that maximizes between-bucket error separation.


In [ ]:
import os
import re
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

from analysis.tsfm_error_analysis import (
    list_tsfms,
    load_tsfm_window_dataframe,
    add_quantile_buckets,
    compute_feature_combo_error_stats,
    build_error_interval_feature_stats,
    plot_feature_pair_heatmaps,
    plot_error_bucket_feature_heatmap,
)

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False


def interval_sort_key(x):
    """Sort bucket labels like '(0.1, 0.3]' by left boundary."""
    s = str(x)
    m = re.match(r"[\[(]\s*([-+]?\d*\.?\d+(?:e[-+]?\d+)?)\s*,", s, flags=re.IGNORECASE)
    if m:
        try:
            return float(m.group(1))
        except Exception:
            return float('inf')
    if s == 'single_bucket':
        return -float('inf')
    return float('inf')


def sort_bucket_index(idx_like):
    vals = [str(v) for v in idx_like]
    return sorted(vals, key=interval_sort_key)


In [ ]:
# ====== Configuration ======
OUTPUT_ROOT = 'correction_datasets_double_res_0'
TSFM_NAME = None  # e.g., 'chronos_bolt_tiny'; None -> auto-select first

# Optional scope filters
DATASET_FILTERS = None
FREQ_FILTERS = None
DOMAIN_FILTERS = None

# Outlier guard controls
OUTLIER_GUARD_ENABLE = True
OUTLIER_METHOD = 'winsor_iqr'   # 'winsor_iqr' | 'winsor_quantile'
OUTLIER_Q_LOW = 0.005
OUTLIER_Q_HIGH = 0.995
OUTLIER_IQR_K = 3.0
SMAPE_HARD_RANGE = (0.0, 200.0)

# Bucket params
FEATURE_BINS = 5
ERROR_BINS = 6
ERROR_METRIC = 'smape'
MIN_BUCKET_COUNT = 30

all_tsfms = list_tsfms(OUTPUT_ROOT)
if not all_tsfms:
    raise RuntimeError(f'No TSFM folders found under: {OUTPUT_ROOT}')
if TSFM_NAME is None:
    TSFM_NAME = all_tsfms[0]

print('Available TSFMs:', all_tsfms)
print('Current TSFM:', TSFM_NAME)


In [ ]:
# ====== Load and basic filtering ======
df = load_tsfm_window_dataframe(OUTPUT_ROOT, TSFM_NAME)
if df.empty:
    raise RuntimeError(f'No rows loaded for TSFM: {TSFM_NAME}')

if DATASET_FILTERS:
    df = df[df['dataset'].isin(DATASET_FILTERS)].copy()
if FREQ_FILTERS:
    df = df[df['freq'].isin(FREQ_FILTERS)].copy()
if DOMAIN_FILTERS:
    df = df[df['domain'].isin(DOMAIN_FILTERS)].copy()
if df.empty:
    raise RuntimeError('Data became empty after scope filtering.')

scope_datasets = sorted(df['dataset'].astype(str).unique().tolist())
print('Raw sample count:', len(df))
print('Datasets in scope:', len(scope_datasets))
print('Frequencies in scope:', sorted(df['freq'].astype(str).unique().tolist()))
print('Domains in scope:', sorted(df['domain'].astype(str).unique().tolist()))
display(df.head(3))


In [ ]:
# ====== Feature set + robust outlier guard ======
default_feature_cols = [
    'hist_mean', 'hist_std', 'hist_range_p10_p90',
    'trend_slope', 'trend_strength', 'remainder_strength',
    'fft_low_ratio', 'fft_high_ratio', 'fft_dominant_freq',
]
feature_cols = [c for c in default_feature_cols if c in df.columns]
print('Feature columns:', feature_cols)


def winsor_by_iqr(s: pd.Series, k: float = 3.0):
    s_num = pd.to_numeric(s, errors='coerce')
    q1, q3 = s_num.quantile(0.25), s_num.quantile(0.75)
    iqr = q3 - q1
    if pd.isna(iqr) or iqr <= 0:
        return s_num
    lo, hi = q1 - k * iqr, q3 + k * iqr
    return s_num.clip(lower=lo, upper=hi)


def winsor_by_quantile(s: pd.Series, ql: float = 0.005, qh: float = 0.995):
    s_num = pd.to_numeric(s, errors='coerce')
    lo, hi = s_num.quantile(ql), s_num.quantile(qh)
    return s_num.clip(lower=lo, upper=hi)


def build_analysis_df(df_raw: pd.DataFrame, feat_cols, metric='smape'):
    out = df_raw.copy().replace([np.inf, -np.inf], np.nan)
    out[metric] = pd.to_numeric(out[metric], errors='coerce')

    core_cols = [metric] + feat_cols
    out = out.dropna(subset=core_cols)
    out = out[(out[metric] >= SMAPE_HARD_RANGE[0]) & (out[metric] <= SMAPE_HARD_RANGE[1])]

    cols_to_guard = [metric] + feat_cols
    for c in cols_to_guard:
        if OUTLIER_METHOD == 'winsor_quantile':
            out[c] = winsor_by_quantile(out[c], ql=OUTLIER_Q_LOW, qh=OUTLIER_Q_HIGH)
        else:
            out[c] = winsor_by_iqr(out[c], k=OUTLIER_IQR_K)

    out = out.dropna(subset=cols_to_guard)
    return out


df_analysis = build_analysis_df(df, feature_cols, metric=ERROR_METRIC) if OUTLIER_GUARD_ENABLE else df.copy()
removed = len(df) - len(df_analysis)
removed_ratio = removed / max(len(df), 1)

scope_text = (
    f"datasets={df_analysis['dataset'].nunique()} | samples={len(df_analysis)} | "
    f"removed={removed} ({removed_ratio:.2%})"
)

print('Outlier guard enabled:', OUTLIER_GUARD_ENABLE)
print('Outlier method:', OUTLIER_METHOD)
print('Before:', len(df), 'After:', len(df_analysis))

# bucketed frame for downstream charts
df_b = add_quantile_buckets(df_analysis, feature_cols=feature_cols, n_bins=FEATURE_BINS)


## Figure 1 — Global Error Distribution + Top-Dataset Boxplot

**Figure name**: Global sMAPE distribution and Top-10 dataset comparison.  

**Left panel**
- **X-axis**: sMAPE value.
- **Y-axis**: sample count.
- **Meaning**: global error shape (tail thickness, skewness, concentration).

**Right panel**
- **X-axis**: dataset name (Top-10 by sample count).
- **Y-axis**: sMAPE.
- **Meaning**: dataset-level spread and median differences.

**How to analyze**
1. Check whether the left panel still has a heavy tail after outlier guard.
2. In right panel, compare medians/IQRs to identify hard datasets.
3. Cross-check hard datasets against domain/frequency for root-cause analysis.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 6), gridspec_kw={'width_ratios': [1.1, 1.1]})

sns.histplot(df_analysis[ERROR_METRIC], bins=50, kde=True, ax=axes[0], color='#4C72B0')
axes[0].set_title(f"{TSFM_NAME} | Global {ERROR_METRIC.upper()} Distribution | {scope_text}")
axes[0].set_xlabel(ERROR_METRIC.upper())
axes[0].set_ylabel('Count')

# Top-10 datasets by sample size
top_ds = df_analysis['dataset'].value_counts().head(10).index
box_df = df_analysis[df_analysis['dataset'].isin(top_ds)].copy()
order = box_df.groupby('dataset')[ERROR_METRIC].median().sort_values(ascending=False).index
sns.boxplot(data=box_df, x='dataset', y=ERROR_METRIC, order=order, ax=axes[1], color='#55A868')
axes[1].set_title(f"{TSFM_NAME} | Top-10 Datasets {ERROR_METRIC.upper()} Boxplot | {scope_text}")
axes[1].tick_params(axis='x', rotation=40)
axes[1].set_xlabel('Dataset')
axes[1].set_ylabel(ERROR_METRIC.upper())

plt.tight_layout()
plt.show()


## Figure 2 — Per-Feature Bucket Profile (One Figure per Feature)

**Figure name**: Feature bucket profile for sMAPE.  

**Axes**
- **X-axis**: bucket intervals of one feature (sorted by interval left boundary).
- **Left Y-axis**: `mean/max` sMAPE and `mean ± std` band.
- **Right Y-axis**: sample count in each bucket.

**Other elements**
- Bar: mean sMAPE.
- Red line: max sMAPE.
- Purple band: mean ± std.
- Green dashed line: bucket count.

**How to analyze**
1. Trend shape reveals monotonic/non-monotonic feature-error relationship.
2. Count line helps avoid over-interpreting sparse buckets.
3. Compare features to identify strong explanatory signals.


In [ ]:
def plot_feature_smape_profile(df_bucketed, feature, metric='smape'):
    bucket_col = f'{feature}_bucket'
    g = (
        df_bucketed.groupby(bucket_col, observed=True)[metric]
        .agg(['mean', 'max', 'std', 'count'])
        .reset_index()
    )

    g['bucket_str'] = g[bucket_col].astype(str)
    g = g.sort_values('bucket_str', key=lambda s: s.map(interval_sort_key)).reset_index(drop=True)

    fig, ax1 = plt.subplots(figsize=(14, 5.5))
    x = np.arange(len(g))

    ax1.bar(x, g['mean'], color='#4C72B0', alpha=0.85, width=0.75, label=f'{metric.upper()} Mean')
    ax1.plot(x, g['max'], color='#C44E52', marker='o', linewidth=2.0, label=f'{metric.upper()} Max')
    std = g['std'].fillna(0.0)
    ax1.fill_between(x, g['mean'] - std, g['mean'] + std, color='#8172B2', alpha=0.2, label=f'{metric.upper()} ±1STD')

    ax1.set_xticks(x)
    ax1.set_xticklabels(g['bucket_str'], rotation=30, ha='right')
    ax1.set_xlabel(f'{feature} buckets')
    ax1.set_ylabel(metric.upper())
    ax1.set_title(f"TSFM={TSFM_NAME} | Metric={metric.upper()} | Feature={feature} | {scope_text}")

    ax2 = ax1.twinx()
    ax2.plot(x, g['count'], color='#55A868', marker='s', linestyle='--', linewidth=1.8, label='Count')
    ax2.set_ylabel('Count')

    # Move legend outside to avoid line overlap
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc='upper left', bbox_to_anchor=(1.01, 1.0), borderaxespad=0.0)

    plt.tight_layout(rect=[0, 0, 0.88, 1])
    plt.show()

for feat in feature_cols:
    plot_feature_smape_profile(df_b, feat, metric=ERROR_METRIC)


## Figure 3 — Two-Feature Bucket Heatmaps

**Figure name**: Two-feature bucket heatmaps (`mean`, `max`, `std` of sMAPE).  

**Axes**
- **X-axis / Y-axis**: bucket intervals of two selected features.
- **Color**: the selected statistic value.

**How features are selected**
- We pick the top-2 features by absolute correlation with sMAPE.

**How to analyze**
1. Identify hot regions (high error zones) in the 2D feature space.
2. Compare mean/max/std panels to distinguish stable vs volatile high-error regions.
3. Use these regions as candidate rule-based bucket definitions.


In [ ]:
if len(feature_cols) >= 2:
    corr = df_analysis[feature_cols + [ERROR_METRIC]].corr(numeric_only=True)[ERROR_METRIC].drop(ERROR_METRIC)
    top2 = corr.abs().sort_values(ascending=False).head(2).index.tolist()

    stats = compute_feature_combo_error_stats(df_b, top2, error_metric=ERROR_METRIC)

    # Ensure bucket ordering is by interval position
    x_col = f'{top2[0]}_bucket'
    y_col = f'{top2[1]}_bucket'
    stats[x_col] = stats[x_col].astype(str)
    stats[y_col] = stats[y_col].astype(str)
    stats = stats.sort_values([y_col, x_col], key=lambda s: s.map(interval_sort_key)).reset_index(drop=True)

    fig = plot_feature_pair_heatmaps(stats, x_col, y_col, figsize=(22, 6))
    fig.suptitle(
        f"TSFM={TSFM_NAME} | Metric={ERROR_METRIC.upper()} | 2D Bucket Heatmaps ({top2[0]} × {top2[1]}) | {scope_text}",
        y=1.05,
    )
    plt.show()



## Figure 4 — Error-Bucket-to-Feature Heatmaps

**Figure name**: Feature statistics conditioned on error buckets.  

**Axes**
- **X-axis**: error bucket intervals (sorted).
- **Y-axis**: feature names.
- **Color**: feature `mean` / `max` / `std` within each error bucket.

**How to analyze**
1. Observe how feature distributions shift from low-error to high-error buckets.
2. Features with strong monotonic drift are strong candidates for bucket rules.
3. Cross-check with Figure 2 and Figure 3 for consistency.


In [ ]:
err_feat_stats = build_error_interval_feature_stats(
    df_analysis,
    error_metric=ERROR_METRIC,
    feature_cols=feature_cols,
    n_error_bins=ERROR_BINS,
)

# sort error buckets
if not err_feat_stats.empty and 'error_bucket' in err_feat_stats.columns:
    err_feat_stats['error_bucket'] = err_feat_stats['error_bucket'].astype(str)
    err_feat_stats = err_feat_stats.sort_values('error_bucket', key=lambda s: s.map(interval_sort_key)).reset_index(drop=True)

display(err_feat_stats.head())

for stat in ['mean', 'max', 'std']:
    fig = plot_error_bucket_feature_heatmap(err_feat_stats, feature_cols, stat=stat, figsize=(16, 6.5))
    fig.suptitle(
        f"TSFM={TSFM_NAME} | Metric={ERROR_METRIC.upper()} | Error-Bucket Feature {stat.capitalize()} | {scope_text}",
        y=1.02
    )
    plt.show()


## Methodology (Part II): How We Search for a Better Bucket-Based Classification Rule

Goal: find a feature-based bucketing rule that maximizes **between-bucket error separation** while keeping buckets statistically reliable.

### Step-by-step computation flow

1. **Candidate generation**
   - Single-feature bucket schemes.
   - Two-feature combined bucket schemes.
   - Multiple bin counts per scheme.

2. **Bucket construction**
   - For each feature in a scheme, apply quantile bucketing (`qcut`).
   - Combine feature buckets into one joint bucket label.

3. **Reliability constraint**
   - Discard buckets with sample count `< MIN_BUCKET_COUNT`.
   - Discard schemes with fewer than 2 valid buckets.

4. **Separation statistics**
   - Compute group means and counts.
   - Compute:
     - `SS_total` = total variance of sMAPE,
     - `SS_between` = between-bucket explained variance,
     - `eta2 = SS_between / SS_total`.
   - Compute `mean_gap = max(bucket_mean) - min(bucket_mean)`.

5. **Composite objective**

\[
	ext{score} = \eta^2 \cdot \log(1 + 	ext{mean\_gap})
\]

Interpretation:
- `eta2` rewards global explanatory power;
- `mean_gap` rewards practical separation margin;
- log dampens extreme gap instability.

6. **Ranking & selection**
   - Rank all candidate schemes by `score` descending.
   - Inspect top results and choose the most interpretable high-score scheme.


In [ ]:
def evaluate_bucket_scheme(df_in, features, n_bins=5, metric='smape', min_count=30):
    tmp = df_in.copy()

    bucket_cols = []
    for f in features:
        col = f'{f}_bucket_opt'
        tmp[col] = pd.qcut(tmp[f], q=n_bins, duplicates='drop')
        bucket_cols.append(col)

    combo_col = '_combo_bucket_'
    tmp[combo_col] = tmp[bucket_cols].astype(str).agg(' | '.join, axis=1)

    g = tmp.groupby(combo_col, observed=True)[metric].agg(['mean', 'count'])
    g = g[g['count'] >= min_count]
    if len(g) < 2:
        return None

    valid_buckets = set(g.index.tolist())
    work = tmp[tmp[combo_col].isin(valid_buckets)].copy()
    overall_mean = work[metric].mean()

    ss_total = np.sum((work[metric] - overall_mean) ** 2)
    ss_between = 0.0
    for _, row in g.iterrows():
        ss_between += row['count'] * (row['mean'] - overall_mean) ** 2

    eta2 = float(ss_between / (ss_total + 1e-12))
    mean_gap = float(g['mean'].max() - g['mean'].min())
    score = float(eta2 * np.log1p(mean_gap))

    return {
        'features': tuple(features),
        'n_bins': int(n_bins),
        'num_valid_buckets': int(len(g)),
        'min_bucket_count': int(g['count'].min()),
        'max_bucket_count': int(g['count'].max()),
        'eta2': eta2,
        'mean_gap': mean_gap,
        'score': score,
    }

results = []
for f in feature_cols:
    for b in [3, 4, 5, 6]:
        r = evaluate_bucket_scheme(df_analysis, [f], n_bins=b, metric=ERROR_METRIC, min_count=MIN_BUCKET_COUNT)
        if r is not None:
            results.append(r)

for f1, f2 in itertools.combinations(feature_cols, 2):
    for b in [3, 4, 5]:
        r = evaluate_bucket_scheme(df_analysis, [f1, f2], n_bins=b, metric=ERROR_METRIC, min_count=MIN_BUCKET_COUNT)
        if r is not None:
            results.append(r)

res_df = pd.DataFrame(results).sort_values('score', ascending=False).reset_index(drop=True)
print('Candidate scheme count:', len(res_df))
display(res_df.head(20))


## Figure 5 — Best Bucket Scheme Visualization

**Figure name**: Best scheme bucket boxplot and bucket statistics table.  

**Axes**
- **X-axis**: final bucket labels from the selected best scheme.
- **Y-axis**: sMAPE.

**Other elements**
- Display table: bucket-level `mean/std/count` sorted by mean.
- Boxplot: distribution spread per final bucket.

**How to analyze**
1. Large gaps between bucket medians indicate actionable separability.
2. Check counts to ensure separation is not from tiny buckets.
3. Use the selected feature(s) and bucket ranges as a deployable classification policy.


In [ ]:
if res_df.empty:
    raise RuntimeError('No valid scheme found. Consider lowering MIN_BUCKET_COUNT.')

best = res_df.iloc[0].to_dict()
best_features = list(best['features'])
best_bins = int(best['n_bins'])

print('Best scheme:')
print(best)

work = df_analysis.copy()
for f in best_features:
    work[f'{f}_bucket_best'] = pd.qcut(work[f], q=best_bins, duplicates='drop')
work['best_bucket'] = work[[f'{f}_bucket_best' for f in best_features]].astype(str).agg(' | '.join, axis=1)

bucket_stats = (
    work.groupby('best_bucket', observed=True)[ERROR_METRIC]
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
bucket_stats = bucket_stats.sort_values('best_bucket', key=lambda s: s.map(interval_sort_key)).reset_index(drop=True)
display(bucket_stats)

order = bucket_stats['best_bucket'].tolist()
plt.figure(figsize=(max(12, min(24, len(order) * 0.9)), 6.5))
sns.boxplot(data=work[work['best_bucket'].isin(order)], x='best_bucket', y=ERROR_METRIC, order=order, color='#4C72B0')
plt.xticks(rotation=35, ha='right')
plt.title(
    f"Best Bucket Scheme | TSFM={TSFM_NAME} | Metric={ERROR_METRIC.upper()} | "
    f"Features={best_features} | bins={best_bins} | {scope_text}"
)
plt.xlabel('Best Buckets')
plt.ylabel(ERROR_METRIC.upper())
plt.tight_layout()
plt.show()
